# C3 · Introducción a Linux y Bash para datos biológicos

**Curso:** Bioinformática y Biología Computacional · Universidad EAFIT  
**Duración sugerida:** 3 horas  
**Modalidad:** explicación breve → práctica guiada → reto → evidencia reproducible

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UniversidadEAFIT/compubiol_course/blob/master/notebooks/03_linux/03_intro_linux.ipynb)

> **Continuidad del material histórico:** esta versión reemplaza y amplía `20231/IntroLinux/Intro_Linux.ipynb`.

## Pregunta guía

Un FASTA tiene encabezados heterogéneos. **¿Cómo responder preguntas y producir una versión normalizada sin abrir el archivo en Excel ni editarlo manualmente?**

### Objetivos

- navegar con rutas absolutas y relativas;
- crear, copiar, mover e inspeccionar archivos de forma segura;
- combinar `grep`, `cut`, `sort`, `uniq`, `wc`, `head`, `tail` y `awk`;
- usar redirecciones, pipes y expresiones regulares;
- documentar una solución como comandos reproducibles.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import importlib.util

REPO_URL = "https://github.com/UniversidadEAFIT/compubiol_course.git"
COLAB_DIR = Path("/content/compubiol_course")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB and importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

if IN_COLAB and not COLAB_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_DIR)], check=True)
    os.chdir(COLAB_DIR)

start = Path.cwd().resolve()
ROOT = next((p for p in [start, *start.parents] if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No se encontró la raíz del curso. Ejecute el notebook desde el repositorio clonado."
    )
os.chdir(ROOT)
os.environ["COURSE_ROOT"] = str(ROOT)
print(f"Raíz del curso: {ROOT}")

## 1. Modelo mental

La terminal ejecuta programas sobre argumentos. El sistema de archivos es un árbol: `/` es la raíz; `.` es el directorio actual; `..` es el padre; `~` es el hogar del usuario.

```bash
pwd
ls -lah
find data/module03 -maxdepth 1 -type f
```

En Windows use WSL2/Ubuntu para reproducir el entorno Unix. No se requiere una distribución histórica especializada como BioLinux.

In [ ]:
import subprocess, os
subprocess.run(["bash", "-lc", 'pwd; printf "\\nArchivos C3:\\n"; ls -lah "$COURSE_ROOT/data/module03"'], check=True)

## 2. Inspección rápida

Nunca suponga el formato por la extensión. Examine las primeras líneas, el tamaño y caracteres delimitadores.

In [ ]:
%%bash
set -euo pipefail
FASTA="$COURSE_ROOT/data/module03/sequences_mixed_headers.fasta"
TSV="$COURSE_ROOT/data/module03/samples.tsv"
printf '%s\n' '--- FASTA ---'
head -n 6 "$FASTA"
printf '%s\n' '--- TSV ---'
head -n 4 "$TSV" | column -t -s $'\t' 2>/dev/null || head -n 4 "$TSV"

## 3. Preguntas como pipelines

In [ ]:
%%bash
set -euo pipefail
FASTA="$COURSE_ROOT/data/module03/sequences_mixed_headers.fasta"

printf 'Número de secuencias: '
grep -c '^>' "$FASTA"

printf '\nEncabezados que mencionan Escherichia_coli:\n'
grep '^>' "$FASTA" | grep -i 'taxon=Escherichia_coli'

printf '\nIDs normalizados y frecuencia:\n'
grep '^>' "$FASTA" \
  | sed 's/^>//' \
  | sed -E 's/[ |].*$//' \
  | tr '[:lower:]' '[:upper:]' \
  | sort \
  | uniq -c

Un pipe `|` conecta la salida estándar de un comando con la entrada del siguiente. La redirección `>` crea o reemplaza un archivo; `>>` añade. Antes de usar `>`, confirme el destino.

In [ ]:
%%bash
set -euo pipefail
mkdir -p "$COURSE_ROOT/results/module03"
TSV="$COURSE_ROOT/data/module03/samples.tsv"
cut -f2 "$TSV" | tail -n +2 | sort | uniq -c | sort -nr \
  > "$COURSE_ROOT/results/module03/taxon_counts.txt"
cat "$COURSE_ROOT/results/module03/taxon_counts.txt"

## 4. Longitudes FASTA con AWK

In [ ]:
%%bash
set -euo pipefail
FASTA="$COURSE_ROOT/data/module03/sequences_mixed_headers.fasta"
awk '
  /^>/ {
    if (id != "") print id, seqlen
    id=$0; sub(/^>/,"",id); sub(/[ |].*/,"",id); seqlen=0; next
  }
  { gsub(/[[:space:]]/,"",$0); seqlen += length($0) }
  END { if (id != "") print id, seqlen }
' OFS='\t' "$FASTA"

### Checkpoint

1. ¿Por qué `grep -c '^>'` cuenta registros FASTA pero no bases?  
2. ¿Qué diferencia hay entre `grep 'seq'` y `grep '^>seq'`?  
3. ¿Qué parte del pipeline detecta que `seq002` está duplicado?

## 5. Normalización sin perder procedencia

In [ ]:
%%bash
set -euo pipefail
IN="$COURSE_ROOT/data/module03/sequences_mixed_headers.fasta"
OUT="$COURSE_ROOT/results/module03/normalized_headers.fasta"

awk '
  /^>/ {
    raw=substr($0,2)
    id=raw; sub(/[ |].*/,"",id); id=toupper(id)
    taxon="NA"; gene="NA"; sample="NA"
    if (match(tolower(raw), /taxon=[^ |]+/)) taxon=substr(raw,RSTART+6,RLENGTH-6)
    if (match(tolower(raw), /gene=[^ |]+/)) gene=substr(raw,RSTART+5,RLENGTH-5)
    if (match(tolower(raw), /sample=[^ |]+/)) sample=substr(raw,RSTART+7,RLENGTH-7)
    print ">" id "|taxon=" taxon "|gene=" gene "|sample=" sample
    next
  }
  { print toupper($0) }
' "$IN" > "$OUT"

head -n 6 "$OUT"

## Seguridad básica

- Cite variables: `"$file"`, no `$file`.
- Use `rm -i` mientras aprende y revise `pwd`/`ls` antes de borrar.
- No use `sudo` para resolver errores de permisos dentro de un proyecto.
- No escriba sobre el archivo original.
- Conserve comandos en `commands.sh` y active `set -euo pipefail` en scripts.

## Reto

Genere una tabla `id, taxon, gene, sample, length`, detecte duplicados y explique dos limitaciones del parser. Una solución robusta debe declarar qué ocurre con encabezados que no cumplen el patrón.